In [1]:
import os.path as op
import matplotlib
import matplotlib.pyplot as plt
import mne
import numpy as np
import joblib
from mne.beamformer import apply_lcmv, make_lcmv
from mne import make_forward_solution, setup_source_space, setup_volume_source_space

In [6]:
path = '/Users/immlab/Desktop/IMM-Lab'
meg_path = op.join(path, 'MEG')
mri_path = op.join(path, 'new_MRI')

In [7]:
mne_fsavg = mne.datasets.fetch_fsaverage(subjects_dir = mri_path, verbose=True)

179 files missing from root.txt in /Users/immlab/Desktop/IMM-Lab/new_MRI


100%|████████████████████████████████████████| 196M/196M [00:00<00:00, 212GB/s]


Extracting missing files
Successfully extracted 179 files
10 files missing from bem.txt in /Users/immlab/Desktop/IMM-Lab/new_MRI/fsaverage


100%|████████████████████████████████████████| 239M/239M [00:00<00:00, 152GB/s]


Extracting missing files
Successfully extracted 10 files


In [ ]:
#Create a source space including cerebellum for fsaverage: 

average_sub = subj = 'fsaverage' #Change this to vml_avg_child to setup averaged volume src for children
bem_dir = op.join(mri_path, subj, 'bem')
fname_aseg = op.join(mri_path, subj, 'mri', 'aseg.mgz')

src_to = mne.read_source_spaces(op.join(mri_path, average_sub, 'bem', f'{average_sub}-ico-5-src.fif'))#change to oct6 for vml_avg_child

model = mne.make_bem_model(subject=subj, ico=5, conductivity=(0.3,),
                           subjects_dir=mri_path)

mne.write_bem_surfaces(op.join(mri_path, subj, 'bem', f'{subj}-bem-model.fif'), model, overwrite=True)

fname_model = op.join(bem_dir, f"{subj}-bem-model.fif")

labels_vol = [ 
    'Right-Cerebellum-Cortex',
    'Right-Cerebellum-White-Matter',
    'Left-Cerebellum-Cortex',
    'Left-Cerebellum-White-Matter'
]

fs_avg_vol_src = setup_volume_source_space(
    subj,
    mri = fname_aseg,
    pos = 5.0,
    bem = fname_model, 
    add_interpolator = True,
    volume_label = labels_vol,
    subjects_dir = mri_path
)

src_to += fs_avg_vol_src
src_to.save(op.join(mri_path, subj, 'bem', f'{subj}-mixed-src.fif'), overwrite=True)
src_to.plot(subjects_dir=mri_path)